# 01 · KG modes — auto-built knowledge graph + dual-level retrieval

LightRAG extracts a knowledge graph from raw text (entities + relationships, by LLM) and answers with **dual-level retrieval**. This notebook reads the `lightrag_wiki` graph built by `build.py` and runs the same question through all five query modes.

> Run `build.py` first.

In [1]:
import sys, pathlib, logging
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # examples/demos
# LightRAG resets its own logger to INFO on construction, so silence verbose
# INFO/WARNING logs globally (logging.disable can't be overridden by setLevel).
logging.disable(logging.WARNING)
from _common import config
from _common.rag import build_rag
from lightrag import QueryParam
from lightrag.kg.shared_storage import initialize_pipeline_status
config.require_openai_key()
rag = build_rag("lightrag_wiki")
await rag.initialize_storages(); await initialize_pipeline_status()
g = rag.chunk_entity_relation_graph
print("entities in graph:", f"{len(await g.get_all_labels()):,}")
print("most-connected:", ", ".join(await g.get_popular_labels(limit=10)))

entities in graph: 15,474
most-connected: Armenia, United States, Angola, Animated Television Series, Academy Award for Best Production Design, Andhra Pradesh, Apple II, Abydos, All Souls' Day, Apiaceae


## The five modes

`naive` = chunks only (baseline); `local` = entity-centric (low-level keywords); `global` = relationship/theme (high-level keywords); `hybrid` = local+global; `mix` = hybrid KG + chunks (default).

In [2]:
question = "Tell me about Armenia: its key facts and the people, places, and events associated with it."
for mode in ["naive", "local", "global", "hybrid", "mix"]:
    ans = await rag.aquery(question, QueryParam(mode=mode, enable_rerank=False))
    print(f"\n### {mode}\n" + str(ans).strip()[:500] + " …")


### naive
Armenia, a landlocked country located in the South Caucasus region, is characterized by its rich history, rugged terrain, and vibrant culture. Below are some key facts about Armenia, along with information about its people, places, and significant historical events.

### Key Facts

- **Geography**: Armenia is bordered by Georgia to the north, Azerbaijan to the east, Iran to the south, and Turkey to the west. The country occupies a mountainous terrain with elevations that rise sharply, including  …



### local
### Key Facts about Armenia

- **Location**: Armenia is a landlocked country in the South Caucasus, bordered by Georgia to the north, Azerbaijan to the east, Iran to the south, and Turkey to the west. 

- **Geography**: The country features a mountainous terrain and possesses a highland continental climate, with most of its area exceeding 1,600 meters above sea level. The highest point is Mount Aragats, which rises to 4,090 meters. Significant rivers include the Araks and Debed Rivers.

- **Popu …



### global
Armenia is a landlocked country situated in the South Caucasus region of Eurasia, bordered by Georgia to the north, Azerbaijan (and the Nakhchivan exclave) to the east, Iran to the south, and Turkey to the west. Here are some key facts about Armenia, along with notable people, places, and events associated with the country:

### Key Facts

- **Geography**: Armenia is characterized by its mountainous terrain and highland continental climate. The Armenian Plateau is a significant geographical feat …



### hybrid
### Key Facts about Armenia

- **Location**: Armenia is a landlocked country situated in the South Caucasus region of Eurasia. It is bordered by Georgia to the north, Azerbaijan to the east, Iran to the south, and Turkey to the west. 
- **Geography**: The country features a mountainous terrain and has a highland continental climate, which contributes to its ecological diversity. The Armenian Plateau and the Lesser Caucasus mountain range are significant geographical features. The highest point i …



### mix
### Key Facts about Armenia

- **Geographical Location**: Armenia is a landlocked country located in the South Caucasus region of Eurasia. It is bordered by Georgia to the north, Azerbaijan to the east (including the Nakhchivan exclave), Iran to the south, and Turkey to the west. The country is characterized by its mountainous terrain and highland continental climate, which contributes to its unique landscape and ecological diversity. Mount Aragats is the highest point in Armenia.

- **Cultural  …


## The dual-level signal (`aquery_data`)

The retrieved structure makes the difference visible: `local` leans on low-level (entity) keywords, `global` on high-level (theme) keywords, `mix` pulls entities + relationships + raw chunks.

In [3]:
for mode in ["local", "global", "mix"]:
    d = await rag.aquery_data(question, QueryParam(mode=mode, enable_rerank=False))
    data, kw = d.get("data", {}), d.get("metadata", {}).get("keywords", {})
    print(f"{mode:7} entities={len(data.get('entities',[])):3} "
          f"relationships={len(data.get('relationships',[])):3} chunks={len(data.get('chunks',[])):3} "
          f"| high-level={kw.get('high_level')} low-level={kw.get('low_level')}")

local   entities= 40 relationships=118 chunks= 20 | high-level=['Armenia', 'key facts', 'people', 'places', 'events'] low-level=['Armenians', 'Yerevan', 'Ararat', 'cultural heritage', 'history']


global  entities= 41 relationships= 40 chunks= 20 | high-level=['Armenia', 'key facts', 'people', 'places', 'events'] low-level=['Armenia']


mix     entities= 67 relationships=106 chunks= 20 | high-level=['Armenia', 'key facts', 'people', 'places', 'events'] low-level=['Armenia']


## `only_need_context` — what was retrieved, without an LLM answer

In [4]:
ctx = await rag.aquery(question, QueryParam(mode="mix", only_need_context=True))
print(str(ctx).strip()[:1100])

Knowledge Graph Data (Entity):

```json
{"entity": "Armenia", "type": "location", "description": "Armenia is a landlocked country located in the South Caucasus region of Eurasia, bordered by Georgia to the north, Azerbaijan, including the Nakhchivan exclave, to the east, Iran to the south, and Turkey to the west. The country is characterized by its mountainous terrain and highland continental climate, which contributes to its unique landscape and ecological diversity.\n\nArmenia has a rich history and cultural heritage, marked by significant events that have shaped its population and society. Over time, the population has experienced considerable changes due to various historical circumstances, including periods of both growth and decline. The country is recognized for its distinctive cultural contributions, including its language, the Armenian language, which is the official state language.\n\nSince gaining independence from the Soviet Union, Armenia has undergone notable historical a

## Follow-up questions (`conversation_history`)

LightRAG carries multi-turn context, so a follow-up like *“what countries does it border?”* resolves *“it”* from the previous turn.

In [5]:
history = [
    {"role": "user", "content": "Tell me about Armenia."},
    {"role": "assistant", "content": "Armenia is a landlocked country in the South Caucasus."},
]
follow_up = "What countries does it border?"
ans = await rag.aquery(follow_up, QueryParam(mode="mix", enable_rerank=False, conversation_history=history))
print("Follow-up:", follow_up, "\n")
print(str(ans).strip()[:500])

Follow-up: What countries does it border? 

Armenia is bordered by several countries:

- **Georgia** to the north
- **Azerbaijan** to the east and southwest, including the disputed Nagorno-Karabakh region
- **Iran** to the south
- **Turkey** to the west

These borders indicate not only geographical relationships but also historical and cultural connections, particularly with neighboring countries that share complex political dynamics. 

### References

- [8] Geography of Armenia
- [9] Foreign relations of Armenia


## How this graph was built

`build.py` streams Wikipedia and hands batches to `ainsert` — LightRAG chunks each article, extracts entities + relationships with the LLM, and merges them across articles:

```python
rag = build_rag("lightrag_wiki")
await rag.ainsert(texts, ids=ids, file_paths=titles)   # the LLM extracts the KG
```

This notebook reads the finished graph; run `build.py` to (re)build it.

In [6]:
await rag.finalize_storages()